In [67]:
import os
from dotenv import load_dotenv
import ulid
from langfuse import Langfuse, observe
from langfuse.langchain import CallbackHandler
from google import genai
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
# For auto-tracing with Google GenAI
from openinference.instrumentation.google_genai import GoogleGenAIInstrumentor
from langchain_google_genai import ChatGoogleGenerativeAI

In [68]:
load_dotenv()

# Initialize Google Gemini client
google_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Optional: Instrument Google GenAI for auto-tracing
GoogleGenAIInstrumentor().instrument()

model_id = "gemini-3-flash-preview"  # or "gemini-1.5-flash", "gemini-1.5-pro"

model = ChatGoogleGenerativeAI(
    model=model_id,  # or "gemini-1.5-flash", "gemini-1.5-pro"
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    base_url = "https://generativelanguage.googleapis.com/",
    temperature=0.7,
    max_tokens=1000,
)

print(f"Using model: {model_id}")

Attempting to instrument while already instrumented


Using model: gemini-3-flash-preview


In [69]:
# Initialize Langfuse client
langfuse_client = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")
)

def generate_session_id():
    """Generate a unique session ID using TEAM_NAME and ULID."""
    return f"{os.getenv('TEAM_NAME', 'ai-agent-team')}-{ulid.new().str}"

def invoke_langchain(model, prompt, langfuse_handler):
    """Invoke LangChain with the given prompt and Langfuse handler."""
    messages = [HumanMessage(content=prompt)]
    response = model.invoke(
        messages,
        config={"callbacks": [langfuse_handler]},
    )
    return response.content

@observe()
def run_llm_call(session_id, model, prompt):
    """Run a single LangChain invocation and track it in Langfuse."""
    
    langfuse_handler = CallbackHandler()

    messages = [HumanMessage(content=prompt)]

    response = model.invoke(
        messages,
        config={
            "callbacks": [langfuse_handler],
            "metadata": {
                "session_id": session_id,
                "trace_name": "llm-call",
            },
        },
    )

    return response.content

print("✓ Langfuse initialized successfully")
print(f"✓ Public key: {os.getenv('LANGFUSE_PUBLIC_KEY', 'Not set')[:20]}...")
print("✓ Helper functions ready: generate_session_id(), invoke_langchain(), run_llm_call()")

✓ Langfuse initialized successfully
✓ Public key: pk-lf-04a16318-a800-...
✓ Helper functions ready: generate_session_id(), invoke_langchain(), run_llm_call()


In [70]:
session_id = generate_session_id()
print(f"Session ID: {session_id}\n")

response = run_llm_call(session_id, model, "What is the square root of 144?")

print(f"\nInput:    What is the square root of 144?")
print(f"Response: {response}")

langfuse_client.flush()

print(f"\n✓ Trace sent to Langfuse with full token usage and cost data")
print(f"✓ Grouped under session: {session_id}")

Session ID: ai-agent-team-01KP6R69X0XVQQK3VDYYJX9RNV


Input:    What is the square root of 144?
Response: [{'type': 'text', 'text': 'The square root of 144 is **12**.', 'extras': {'signature': 'EvICCu8CAb4+9vvIQRZTu6QWoTGkZz3n10ccnUdLfNE1waLf83/gO0qNILwP+zLkJHhbMtrbDxi7p271v4pJ4mwWXLW4zQr9EMGQHFmmqqmhGTSPIvgmr8hbMC0uYkFOHr4TdNpvE/NKQkC4f/ZfovTfP66qeHBmlSlgTipwNQFHnAdP8PKsB4WQgCiH9EPbur+qApeRCLPKax6v7SgewtebQDzwygjGcZo98TaptVZUO3oGFEVY/Q+PQQZaxyUCE2+S6bR30bse4YZNYsrIeZ8mIgiU0RVgEvHe93cnB3lrzazSUPf/NEEM9HQPIBO0gMVegdCCu/S+1QjaGk9tNbOZITAZ9Weo06xTkEUHlTmlJgK5TInKflplXMUh7dT3ck3Qj2a0QbPd9AHMpz3k7dxxNSKY99qsmsbdyYPdFCI3aAvdOTN0tSlfc3nIOxC76j+08qxBEwkeZh6U3KozSLbyO5oH1lNFnUGnWEdvHMNvbUwirQ=='}}]

✓ Trace sent to Langfuse with full token usage and cost data
✓ Grouped under session: ai-agent-team-01KP6R69X0XVQQK3VDYYJX9RNV


In [71]:
questions = [
    "What is machine learning?",
    "Explain neural networks briefly.",
    "What is the difference between AI and ML?"
]

session_id = generate_session_id()
print(f"Session ID: {session_id}")
print(f"Making {len(questions)} agent calls with Langfuse tracing...\n")

for i, question in enumerate(questions, 1):
    response = run_llm_call(session_id, model, question)
    print(f"Call {i}: {question[:50]}")
    print(f"  Response: {response[:100]}...\n")

langfuse_client.flush()

print("=" * 50)
print(f"✓ All {len(questions)} traces sent to Langfuse!")
print(f"✓ All grouped under session: {session_id}")

Session ID: ai-agent-team-01KP6R6BKYMD9763FKN8GYHPZY
Making 3 agent calls with Langfuse tracing...

Call 1: What is machine learning?
  Response: [{'type': 'text', 'text': 'At its simplest, **Machine Learning (ML)** is a field of Artificial Intelligence (AI) that focuses on building systems that can learn from data, identify patterns, and make decisions with minimal human intervention.\n\nInstead of a human writing a specific set of instructions for a computer to follow, the computer uses **algorithms** to analyze massive amounts of information and "train" itself to perform a task.\n\n---\n\n### 1. How is it different from traditional programming?\n*   **Traditional Programming:** A human writes rules (If A happens, do B). The computer follows those rules exactly.\n*   **Machine Learning:** You give the computer a lot of examples (Data) and the desired result. The computer figures out the rules and patterns itself.\n\n### 2. How does it work? (The Process)\n1.  **Data Collection:** You

In [74]:
from datetime import datetime
from collections import defaultdict

def get_trace_info(session_id: str):
    """Fetch traces for a session_id and aggregate basic statistics."""
    traces = []
    page = 1

    while True:
        response = langfuse_client.api.trace.list(session_id=session_id, limit=100, page=page)
        if not response.data:
            break
        traces.extend(response.data)
        if len(response.data) < 100:
            break
        page += 1

    if not traces:
        return None

    observations = []
    for trace in traces:
        detail = langfuse_client.api.trace.get(trace.id)
        if detail and hasattr(detail, "observations"):
            observations.extend(detail.observations)

    if not observations:
        return None

    sorted_obs = sorted(
        observations,
        key=lambda o: o.start_time if hasattr(o, "start_time") and o.start_time else datetime.min,
    )

    counts = defaultdict(int)
    costs = defaultdict(float)
    total_time = 0.0

    for obs in observations:
        if hasattr(obs, "type") and obs.type == "GENERATION":
            model = getattr(obs, "model", "unknown") or "unknown"
            counts[model] += 1

            if hasattr(obs, "calculated_total_cost") and obs.calculated_total_cost:
                costs[model] += obs.calculated_total_cost

            if hasattr(obs, "start_time") and hasattr(obs, "end_time"):
                if obs.start_time and obs.end_time:
                    total_time += (obs.end_time - obs.start_time).total_seconds()

    first_input = ""
    if sorted_obs and hasattr(sorted_obs[0], "input"):
        inp = sorted_obs[0].input
        if inp:
            first_input = str(inp)[:100]

    last_output = ""
    if sorted_obs and hasattr(sorted_obs[-1], "output"):
        out = sorted_obs[-1].output
        if out:
            last_output = str(out)[:100]

    return {
        "counts": dict(counts),
        "costs": dict(costs),
        "time": total_time,
        "input": first_input,
        "output": last_output,
    }

def print_results(info):
    """Pretty-print the aggregated trace information."""
    if not info:
        print("\nNo traces found for this session_id.\n")
        return

    print("\nTrace Count by Model:")
    for model, count in info["counts"].items():
        print(f"  {model}: {count}")

    print("\nCost by Model:")
    total = 0.0
    for model, cost in info["costs"].items():
        print(f"  {model}: ${cost:.6f}")
        total += cost
    if total > 0:
        print(f"  Total: ${total:.6f}")

    print(f"\nTotal Time: {info['time']:.2f}s")

    if info["input"]:
        print(f"\nInitial Input:\n  {info['input']}")

    if info["output"]:
        print(f"\nFinal Output:\n  {info['output']}")

    print()

# Example usage
session_id_to_check = "ai-agent-team-01KP6R6BKYMD9763FKN8GYHPZY"
info = get_trace_info(session_id_to_check)
print_results(info)


No traces found for this session_id.

